# 05 — Decision Trees

## A completely different way to learn from data

So far we've used gradient descent to adjust numbers (weights).  
Decision trees take a completely different approach: they ask **yes/no questions** about the data.

**The idea:**
- You have a dataset of examples with features and labels.
- The tree learns: *"what single question, applied to this feature, splits the data best?"*
- It repeats this recursively until each leaf is pure (all one class).

Think of it as a flowchart that the algorithm builds automatically from data.

**Pros vs neural nets:**
| | Decision Tree | Neural Net |
|---|---|---|
| Interpretability | ✅ You can read the rules | ❌ Hard to understand |
| Small data | ✅ Works well | ❌ Needs lots of data |
| Complex patterns | ⚠️ Limited | ✅ Excels |
| Images / text | ❌ Poor | ✅ State of the art |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.datasets import load_iris, make_classification
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

## Dataset: Iris flowers

A classic dataset: 150 iris flowers with 4 measurements each.  
Goal: classify into 3 species based on petal and sepal size.

In [ ]:
iris = load_iris()
X, y = iris.data, iris.target
feature_names = iris.feature_names
class_names   = iris.target_names

print(f"Samples: {len(X)}")
print(f"Features: {feature_names}")
print(f"Classes:  {class_names}")
print()
print("First 5 rows:")
for row, label in zip(X[:5], y[:5]):
    print(f"  {row}  → {class_names[label]}")

## Train a Decision Tree

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# max_depth limits how deep the tree grows → prevents overfitting
tree = DecisionTreeClassifier(max_depth=3, random_state=42)
tree.fit(X_train, y_train)

y_pred = tree.predict(X_test)
print(f"Accuracy on test set: {accuracy_score(y_test, y_pred):.1%}")

## Visualise the tree — you can READ what the model learned

In [ ]:
fig, ax = plt.subplots(figsize=(18, 8))
plot_tree(
    tree,
    feature_names=feature_names,
    class_names=class_names,
    filled=True,
    rounded=True,
    fontsize=11,
    ax=ax,
)
ax.set_title("Decision Tree — each box shows the rule and the samples that reach it", fontsize=13)
plt.tight_layout()
plt.savefig("05_tree.png", dpi=120)
plt.show()

## The tree as plain text — fully human-readable rules

In [ ]:
print(export_text(tree, feature_names=list(feature_names)))

## Decision boundary — where does the tree draw the lines?

In [ ]:
# Plot using only 2 features so we can visualise in 2D
FEAT_X, FEAT_Y = 2, 3   # petal length, petal width

X2 = X[:, [FEAT_X, FEAT_Y]]
tree2 = DecisionTreeClassifier(max_depth=3, random_state=42)
tree2.fit(X2, y)

colors = ["#ff9999", "#99cc99", "#99aaff"]

xx, yy = np.meshgrid(
    np.linspace(X2[:, 0].min() - 0.3, X2[:, 0].max() + 0.3, 400),
    np.linspace(X2[:, 1].min() - 0.3, X2[:, 1].max() + 0.3, 400),
)
Z = tree2.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

fig, ax = plt.subplots(figsize=(8, 6))
ax.contourf(xx, yy, Z, alpha=0.4, cmap=plt.cm.RdYlBu)

for cls_idx, (cls_name, color) in enumerate(zip(class_names, ["red", "green", "blue"])):
    mask = y == cls_idx
    ax.scatter(X2[mask, 0], X2[mask, 1], label=cls_name, color=color, edgecolors="k", s=60)

ax.set_xlabel(feature_names[FEAT_X])
ax.set_ylabel(feature_names[FEAT_Y])
ax.set_title("Decision Tree boundary\n(notice the axis-aligned rectangle cuts — no diagonals!)")
ax.legend()
plt.tight_layout()
plt.savefig("05_boundary.png", dpi=120)
plt.show()

print("\nNotice: decision trees always cut parallel to the axes.")
print("Neural networks can make diagonal and curved cuts.")

## Overfitting — the tree that memorised, not learned

In [ ]:
# Unlimited depth: the tree memorises the training data exactly
tree_overfit = DecisionTreeClassifier(max_depth=None, random_state=42)
tree_overfit.fit(X_train, y_train)

print(f"Deep tree — train accuracy: {accuracy_score(y_train, tree_overfit.predict(X_train)):.1%}")
print(f"Deep tree — test  accuracy: {accuracy_score(y_test,  tree_overfit.predict(X_test)):.1%}")
print()
print(f"Shallow tree (depth 3) — train: {accuracy_score(y_train, tree.predict(X_train)):.1%}")
print(f"Shallow tree (depth 3) — test:  {accuracy_score(y_test,  tree.predict(X_test)):.1%}")
print()
print("The deep tree scores 100% on training data but WORSE on new data.")
print("It memorised the examples instead of learning the pattern. That's OVERFITTING.")

## Confusion Matrix

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(ax=ax, colorbar=False, cmap="Blues")
ax.set_title("Confusion Matrix — rows=true class, cols=predicted class")
plt.tight_layout()
plt.savefig("05_confusion.png", dpi=120)
plt.show()

## Key Takeaways

- Decision trees split data using rules: *"if petal length < 2.45 → setosa"*  
- You can read and explain every decision the model makes — great for regulated industries  
- They cut space with axis-aligned rectangles; neural nets can draw any curve  
- **Overfitting**: a too-deep tree memorises training data; `max_depth` prevents this  
- In practice, **Random Forests** (many trees averaged together) are much stronger than one tree

---

Next: back to neural networks, now using PyTorch — the tool that powers real AI.  
→ `06_pytorch_tensors.py`